# 06. Random Forests: El Poder de los Árboles Unidos

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 75 minutos  
**Prerequisitos:** [05. Árboles de Decisión](05-arboles-decision.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Entender el concepto de ensemble learning y bagging
- Implementar Random Forest desde cero usando múltiples árboles
- Calcular y utilizar Out-of-Bag (OOB) error para validación
- Analizar la importancia de features en modelos ensemble
- Comprender cómo reducir varianza sin aumentar bias
- Aplicar Random Forests a problemas de clasificación y regresión

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.datasets import load_iris, load_breast_cancer, make_classification
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Importar utilidades
import sys
sys.path.append('../../shared/utils')
from visualization import plot_decision_boundary
from datasets import load_dataset

np.random.seed(42)
print("✅ Librerías importadas")

---
## 📌 1. Motivación: La Sabiduría de las Multitudes

### El Problema

En el notebook anterior vimos que los árboles de decisión son:
- ✅ **Interpretables** y fáciles de entender
- ✅ **Poderosos** para capturar patrones no-lineales
- ❌ **Inestables**: Pequeños cambios en datos → árbol muy diferente
- ❌ **Propensos a overfitting**: Memorizan ruido del training set

**Ejemplo ilustrativo:**
```
Dataset Original: [🔴🔴🔵🔵🔴🔵] → Árbol A
Dataset +1 muestra: [🔴🔴🔵🔵🔴🔵🔴] → Árbol B (completamente diferente!)
```

### La Solución: Random Forests

**Idea central:** En lugar de confiar en UN árbol, ¡entrenemos MUCHOS árboles y votemos!

```python
Árbol 1: "Es clase A" 🌳
Árbol 2: "Es clase B" 🌳
Árbol 3: "Es clase A" 🌳
Árbol 4: "Es clase A" 🌳
Árbol 5: "Es clase A" 🌳

Votación: A=4, B=1 → Predicción final: A ✅
```

### ¿Por qué funciona?

**Analogía:** Pregunta a 100 personas cuántas canicas hay en un frasco:
- Cada persona puede estar equivocada (alta varianza)
- Pero el **promedio** de sus respuestas suele estar muy cerca del valor real
- Siempre que los errores no estén correlacionados

### Aplicaciones Reales

- 🏥 **Diagnóstico médico**: Clasificar tipos de cáncer
- 💳 **Detección de fraude**: Identificar transacciones sospechosas
- 📈 **Finanzas**: Predicción de defaults en créditos
- 🎮 **Kinect (Xbox)**: Reconocimiento de poses corporales
- 🌾 **Agricultura**: Predicción de rendimiento de cultivos

### La Pregunta Guía

> **¿Cómo hacemos que los árboles sean diferentes entre sí para que sus errores no estén correlacionados?**

---
## 📊 2. Intuición Visual

In [ ]:
# Comparación: Árbol Individual vs Random Forest
# Generar datos con ruido
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0,
                          n_informative=2, n_clusters_per_class=1,
                          flip_y=0.1, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Entrenar modelos
tree = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)

tree.fit(X_train, y_train)
rf.fit(X_train, y_train)

# Crear grid para visualización
h = 0.02
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predicciones
Z_tree = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
Z_rf = rf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Árbol Individual
axes[0].contourf(xx, yy, Z_tree, alpha=0.3, cmap='RdBu')
axes[0].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdBu', edgecolors='k')
axes[0].set_title(f'Árbol Individual\nAccuracy: {tree.score(X_test, y_test):.3f}', fontsize=14)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Random Forest
axes[1].contourf(xx, yy, Z_rf, alpha=0.3, cmap='RdBu')
axes[1].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdBu', edgecolors='k')
axes[1].set_title(f'Random Forest (100 árboles)\nAccuracy: {rf.score(X_test, y_test):.3f}', fontsize=14)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print("\n💡 Observa:")
print("   • El árbol individual tiene fronteras muy irregulares (overfitting)")
print("   • Random Forest suaviza las fronteras (mejor generalización)")
print("   • Random Forest típicamente tiene mayor accuracy en test")

In [ ]:
# Visualizar cómo cambia el accuracy con el número de árboles
n_trees = range(1, 101, 5)
train_scores = []
test_scores = []

for n in n_trees:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)
    train_scores.append(rf.score(X_train, y_train))
    test_scores.append(rf.score(X_test, y_test))

# Plotly
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(n_trees),
    y=train_scores,
    mode='lines+markers',
    name='Train Accuracy',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=list(n_trees),
    y=test_scores,
    mode='lines+markers',
    name='Test Accuracy',
    line=dict(color='red', width=2)
))

fig.update_layout(
    title="Accuracy vs Número de Árboles",
    xaxis_title="Número de Árboles",
    yaxis_title="Accuracy",
    template="plotly_white",
    font=dict(size=12),
    hovermode='x unified'
)

fig.show()

print("\n💡 Observa:")
print("   • El accuracy mejora rápidamente con los primeros árboles")
print("   • Se estabiliza alrededor de 50-100 árboles")
print("   • Más árboles = más cómputo pero eventualmente rendimientos decrecientes")

---
## 🧮 3. Fundamentos Matemáticos

### 📖 Conceptos Clave

#### 1. Ensemble Learning

Combinar múltiples modelos débiles para crear un modelo fuerte.

**Predicción del ensemble:**

$$
\begin{align}
\text{Clasificación: } \hat{y} &= \text{mode}\{h_1(x), h_2(x), ..., h_T(x)\} \tag{1} \\
\text{Regresión: } \hat{y} &= \frac{1}{T} \sum_{t=1}^{T} h_t(x) \tag{2}
\end{align}
$$

Donde $h_t$ es el $t$-ésimo modelo base y $T$ es el número total de modelos.

#### 2. Bagging (Bootstrap Aggregating)

**Bootstrap:** Muestreo con reemplazo del dataset original.

Para cada árbol $t$:
1. Crear dataset bootstrap $D_t$ muestreando $n$ ejemplos con reemplazo de $D$
2. Entrenar árbol $h_t$ en $D_t$
3. Aproximadamente 63.2% de ejemplos únicos, 36.8% repetidos

**Probabilidad de selección:**
$$
P(\text{ejemplo no seleccionado}) = \left(1 - \frac{1}{n}\right)^n \approx e^{-1} \approx 0.368 \tag{3}
$$

#### 3. Random Forests = Bagging + Feature Randomness

En cada **split** de cada árbol:
- Considerar solo $m$ features aleatorias (de las $p$ totales)
- Típicamente: $m = \sqrt{p}$ para clasificación, $m = p/3$ para regresión

**¿Por qué?** Decorrelaciona los árboles. Sin esto, si hay una feature muy fuerte, todos los árboles la usarían primero → alta correlación.

#### 4. Out-of-Bag (OOB) Error

Para cada ejemplo $i$:
- Hay árboles que NO lo vieron en entrenamiento (~37%)
- Usar solo esos árboles para predecir $i$
- OOB error = accuracy promedio sobre todas las predicciones OOB

**Ventaja:** Validación gratis, sin necesidad de un validation set separado.

$$
\text{OOB Error} = \frac{1}{n} \sum_{i=1}^{n} \mathbb{1}\left[y_i \neq \text{majority vote de árboles que no vieron } x_i\right] \tag{4}
$$

#### 5. Feature Importance

Para cada feature $j$, medir cuánto reduce la impureza en promedio:

$$
\text{Importance}(j) = \frac{1}{T} \sum_{t=1}^{T} \sum_{\text{nodos usando } j} \Delta \text{impurity} \tag{5}
$$

Donde $\Delta \text{impurity}$ es la reducción de Gini/entropía al hacer el split.

### Reducción de Varianza

**Teorema clave:** Si tenemos $T$ predictores independientes con varianza $\sigma^2$:

$$
\text{Var}(\text{promedio}) = \frac{\sigma^2}{T} \tag{6}
$$

Con correlación $\rho$:

$$
\text{Var}(\text{promedio}) = \rho\sigma^2 + \frac{1-\rho}{T}\sigma^2 \tag{7}
$$

**Implicación:** 
- Reducir $\rho$ (decorrelación) es crucial
- Por eso Random Forests usa feature randomness

### Ejemplo Numérico

Dataset: 10 ejemplos

**Bootstrap Sample 1:** [1, 2, 2, 4, 5, 5, 7, 8, 9, 10] (no incluye 3, 6)
**Bootstrap Sample 2:** [1, 1, 3, 4, 5, 6, 7, 8, 9, 9] (no incluye 2, 10)

- Ejemplo 3: OOB para árbol 1 (puede usarse para validación)
- Ejemplo 2: OOB para árbol 2

In [ ]:
# Demostración de Bootstrap
n_samples = 10
original_indices = np.arange(n_samples)

print("🎲 Simulación de Bootstrap Sampling:\n")
print(f"Dataset original: {original_indices}\n")

for i in range(5):
    # Bootstrap sample
    bootstrap_indices = np.random.choice(original_indices, size=n_samples, replace=True)
    unique = np.unique(bootstrap_indices)
    oob = np.setdiff1d(original_indices, unique)
    
    print(f"Bootstrap {i+1}:")
    print(f"  Muestras: {bootstrap_indices}")
    print(f"  Únicas: {len(unique)}/{n_samples} ({len(unique)/n_samples*100:.1f}%)")
    print(f"  OOB: {oob}\n")

print("💡 En promedio, ~63% de ejemplos únicos, ~37% son OOB")

---
## 💻 4. Implementación Desde Cero

In [ ]:
from sklearn.tree import DecisionTreeClassifier

class RandomForest:
    """
    Implementación simplificada de Random Forest para clasificación.
    
    Combina bagging con selección aleatoria de features para crear
    un ensemble de árboles de decisión decorrelacionados.
    
    Parameters:
    -----------
    n_estimators : int, default=100
        Número de árboles en el bosque
    max_features : int or str, default='sqrt'
        Número de features a considerar en cada split
        - 'sqrt': sqrt(n_features)
        - 'log2': log2(n_features)
        - int: número específico
    max_depth : int, default=None
        Profundidad máxima de cada árbol
    min_samples_split : int, default=2
        Mínimo de muestras para hacer split
    bootstrap : bool, default=True
        Si usar bootstrap sampling
    oob_score : bool, default=False
        Si calcular OOB score
    random_state : int, default=None
        Semilla para reproducibilidad
    """
    
    def __init__(self, n_estimators=100, max_features='sqrt', max_depth=None,
                 min_samples_split=2, bootstrap=True, oob_score=False, random_state=None):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.bootstrap = bootstrap
        self.oob_score = oob_score
        self.random_state = random_state
        
        # Se inicializan durante fit
        self.trees = []
        self.n_classes = None
        self.n_features = None
        self.oob_score_ = None
        self.feature_importances_ = None
    
    def _get_max_features(self):
        """Calcula el número de features a usar en cada split"""
        if self.max_features == 'sqrt':
            return int(np.sqrt(self.n_features))
        elif self.max_features == 'log2':
            return int(np.log2(self.n_features))
        elif isinstance(self.max_features, int):
            return self.max_features
        else:
            return self.n_features
    
    def fit(self, X, y):
        """
        Entrena el Random Forest.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Training data
        y : array-like, shape (n_samples,)
            Target values
        
        Returns:
        --------
        self : RandomForest
        """
        # Configurar random state
        if self.random_state is not None:
            np.random.seed(self.random_state)
        
        n_samples, self.n_features = X.shape
        self.n_classes = len(np.unique(y))
        
        max_features = self._get_max_features()
        
        # Para OOB score
        if self.oob_score:
            oob_predictions = np.zeros((n_samples, self.n_classes))
            oob_counts = np.zeros(n_samples)
        
        # Entrenar cada árbol
        print(f"🌲 Entrenando {self.n_estimators} árboles...")
        
        for i in range(self.n_estimators):
            # 1. Bootstrap sampling
            if self.bootstrap:
                indices = np.random.choice(n_samples, size=n_samples, replace=True)
                oob_indices = np.setdiff1d(np.arange(n_samples), np.unique(indices))
            else:
                indices = np.arange(n_samples)
                oob_indices = []
            
            X_sample = X[indices]
            y_sample = y[indices]
            
            # 2. Entrenar árbol con feature randomness
            tree = DecisionTreeClassifier(
                max_features=max_features,
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                random_state=self.random_state + i if self.random_state else None
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)
            
            # 3. Calcular OOB predictions si es necesario
            if self.oob_score and len(oob_indices) > 0:
                oob_pred = tree.predict_proba(X[oob_indices])
                oob_predictions[oob_indices] += oob_pred
                oob_counts[oob_indices] += 1
            
            # Progress
            if (i + 1) % 25 == 0:
                print(f"   Progreso: {i+1}/{self.n_estimators} árboles")
        
        # Calcular OOB score
        if self.oob_score:
            # Evitar división por cero
            oob_counts[oob_counts == 0] = 1
            oob_predictions = oob_predictions / oob_counts[:, np.newaxis]
            oob_pred_classes = np.argmax(oob_predictions, axis=1)
            self.oob_score_ = np.mean(oob_pred_classes == y)
            print(f"\n📊 OOB Score: {self.oob_score_:.4f}")
        
        # Calcular feature importances
        self._calculate_feature_importances()
        
        print(f"\n✅ Entrenamiento completado")
        return self
    
    def _calculate_feature_importances(self):
        """Calcula la importancia de cada feature"""
        importances = np.zeros(self.n_features)
        
        for tree in self.trees:
            importances += tree.feature_importances_
        
        # Normalizar
        self.feature_importances_ = importances / self.n_estimators
    
    def predict(self, X):
        """
        Predice clases para X.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Samples to predict
        
        Returns:
        --------
        y_pred : array, shape (n_samples,)
            Predicted classes
        """
        # Obtener predicciones de todos los árboles
        predictions = np.array([tree.predict(X) for tree in self.trees])
        
        # Votación por mayoría
        # Para cada muestra, contar votos y seleccionar la clase más votada
        y_pred = np.apply_along_axis(
            lambda x: np.bincount(x.astype(int)).argmax(),
            axis=0,
            arr=predictions
        )
        
        return y_pred
    
    def predict_proba(self, X):
        """
        Predice probabilidades de clase.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
        
        Returns:
        --------
        proba : array, shape (n_samples, n_classes)
            Probabilidades de cada clase
        """
        # Promediar probabilidades de todos los árboles
        probas = np.zeros((X.shape[0], self.n_classes))
        
        for tree in self.trees:
            probas += tree.predict_proba(X)
        
        return probas / self.n_estimators
    
    def score(self, X, y):
        """Calcula accuracy"""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

print("✅ Clase RandomForest definida")

In [ ]:
# Probar con Iris dataset
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar nuestro Random Forest
rf_custom = RandomForest(
    n_estimators=100,
    max_features='sqrt',
    max_depth=10,
    oob_score=True,
    random_state=42
)

rf_custom.fit(X_train, y_train)

In [ ]:
# Evaluar
y_pred = rf_custom.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"📊 Resultados:")
print(f"   Train Accuracy: {rf_custom.score(X_train, y_train):.4f}")
print(f"   Test Accuracy: {accuracy:.4f}")
print(f"   OOB Score: {rf_custom.oob_score_:.4f}")

print(f"\n📈 Feature Importances:")
for i, importance in enumerate(rf_custom.feature_importances_):
    print(f"   {iris.feature_names[i]}: {importance:.4f}")

---
## 🏭 5. Versión con Framework (Scikit-learn)

In [ ]:
# Scikit-learn Random Forest
rf_sklearn = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    max_depth=10,
    oob_score=True,
    random_state=42,
    n_jobs=-1  # Usar todos los cores
)

rf_sklearn.fit(X_train, y_train)

y_pred_sklearn = rf_sklearn.predict(X_test)
accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)

print("📊 Comparación: Nuestra Implementación vs Scikit-learn\n")
print("="*60)
print(f"{'Métrica':<25} {'Nuestra':<15} {'Scikit-learn':<15}")
print("="*60)
print(f"{'Test Accuracy':<25} {accuracy:<15.4f} {accuracy_sklearn:<15.4f}")
print(f"{'OOB Score':<25} {rf_custom.oob_score_:<15.4f} {rf_sklearn.oob_score_:<15.4f}")
print("="*60)

In [ ]:
# Visualizar Feature Importances
importances = rf_sklearn.feature_importances_
indices = np.argsort(importances)[::-1]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=[iris.feature_names[i] for i in indices],
    y=importances[indices],
    marker=dict(color=importances[indices], colorscale='Viridis'),
    text=[f"{imp:.3f}" for imp in importances[indices]],
    textposition='outside'
))

fig.update_layout(
    title="Feature Importances - Random Forest",
    xaxis_title="Feature",
    yaxis_title="Importance",
    template="plotly_white",
    font=dict(size=12)
)

fig.show()

print("\n💡 Las feature importances nos dicen qué variables son más relevantes para la predicción")

In [ ]:
# Caso de uso: Breast Cancer Dataset (más desafiante)
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42
)

print("🔬 Dataset: Breast Cancer Wisconsin")
print(f"   Samples: {X_cancer.shape[0]}")
print(f"   Features: {X_cancer.shape[1]}")
print(f"   Classes: {len(np.unique(y_cancer))} (Maligno/Benigno)\n")

# Comparar: Árbol Individual vs Random Forest
tree = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)

tree.fit(X_train_c, y_train_c)
rf.fit(X_train_c, y_train_c)

print("📊 Resultados:\n")
print("="*70)
print(f"{'Modelo':<30} {'Train Acc':<20} {'Test Acc':<20}")
print("="*70)
print(f"{'Árbol Individual':<30} {tree.score(X_train_c, y_train_c):<20.4f} {tree.score(X_test_c, y_test_c):<20.4f}")
print(f"{'Random Forest (100 árboles)':<30} {rf.score(X_train_c, y_train_c):<20.4f} {rf.score(X_test_c, y_test_c):<20.4f}")
print("="*70)

print("\n💡 Random Forest generaliza mejor (menor overfitting)")

### Ventajas de Scikit-learn

- ⚡ **Paralelización**: `n_jobs=-1` usa todos los cores del CPU
- 🔧 **Opciones avanzadas**: `class_weight`, `criterion`, `min_impurity_decrease`
- 📊 **Métricas built-in**: `oob_score_`, `feature_importances_`
- 🎯 **Optimizado**: Implementación en Cython, muy rápida
- 🔄 **Integración**: Funciona con pipelines, grid search, etc.

---
## 🎯 6. Ejercicios

### 🟢 Ejercicio 1: Efecto del Número de Árboles

Entrena Random Forests con diferentes números de árboles (10, 50, 100, 200) y compara su accuracy.

In [ ]:
def ejercicio_1():
    """
    Objetivo: Entender el impacto del número de árboles
    
    Instrucciones:
    1. Usa el dataset de Iris (ya cargado)
    2. Entrena Random Forests con n_estimators = [10, 50, 100, 200]
    3. Calcula test accuracy para cada uno
    4. Retorna un diccionario {n_trees: accuracy}
    
    Returns:
    --------
    results : dict
        {10: acc1, 50: acc2, 100: acc3, 200: acc4}
    """
    # TODO: Tu código aquí
    # results = {}
    # for n in [10, 50, 100, 200]:
    #     rf = RandomForestClassifier(n_estimators=n, random_state=42)
    #     ...
    
    pass

# Descomentar para probar
# results = ejercicio_1()
# for n, acc in results.items():
#     print(f"n_estimators={n}: {acc:.4f}")
# print("\n✅ Deberías ver que el accuracy mejora y se estabiliza")

### 🟡 Ejercicio 2: Importancia de Features

Usa el dataset de Breast Cancer para identificar las 5 features más importantes.

In [ ]:
def ejercicio_2():
    """
    Objetivo: Identificar features relevantes
    
    Instrucciones:
    1. Entrena un Random Forest en Breast Cancer dataset
    2. Obtén feature_importances_
    3. Identifica las 5 features con mayor importancia
    4. Retorna sus nombres y valores de importancia
    
    Returns:
    --------
    top_features : list of tuples
        [(feature_name, importance), ...] ordenado descendente
    """
    # TODO: Tu código aquí
    # cancer = load_breast_cancer()
    # X, y = cancer.data, cancer.target
    # rf = RandomForestClassifier(...)
    # ...
    
    pass

# Descomentar para probar
# top_features = ejercicio_2()
# print("🏆 Top 5 Features Más Importantes:\n")
# for i, (name, imp) in enumerate(top_features[:5], 1):
#     print(f"{i}. {name}: {imp:.4f}")

### 🔴 Ejercicio 3: Comparación con Bagging Simple

Random Forest = Bagging + Feature Randomness. Demuestra la importancia de la feature randomness comparando con bagging puro.

In [ ]:
def ejercicio_3():
    """
    Objetivo: Demostrar el valor de feature randomness
    
    Instrucciones:
    1. Usa Breast Cancer dataset
    2. Entrena dos modelos:
       a) BaggingClassifier con DecisionTreeClassifier (sin feature randomness)
       b) RandomForestClassifier (con feature randomness)
    3. Usa n_estimators=100 para ambos
    4. Compara test accuracy y cross-validation score (5-fold)
    5. Visualiza las correlaciones entre predicciones de árboles individuales
    
    Returns:
    --------
    results : dict
        {'bagging_acc': float, 'rf_acc': float, 
         'bagging_cv': float, 'rf_cv': float}
    """
    # TODO: Tu código aquí
    # Pista 1: Usa BaggingClassifier de sklearn.ensemble
    # Pista 2: Para visualizar correlaciones, obtén predicciones de cada árbol
    #          y calcula la matriz de correlación
    
    pass

# Descomentar para probar
# results = ejercicio_3()
# print("📊 Comparación: Bagging vs Random Forest\n")
# print(f"Bagging (sin feature randomness):")
# print(f"  Test Accuracy: {results['bagging_acc']:.4f}")
# print(f"  CV Score: {results['bagging_cv']:.4f}\n")
# print(f"Random Forest (con feature randomness):")
# print(f"  Test Accuracy: {results['rf_acc']:.4f}")
# print(f"  CV Score: {results['rf_cv']:.4f}\n")
# print("💡 Random Forest debería tener menor correlación entre árboles")

---
## 📚 7. Resumen y Recursos

### 🎯 Puntos Clave

1. **Random Forest = Bagging + Feature Randomness**
   - Múltiples árboles entrenados en bootstrap samples
   - Cada split considera solo un subset aleatorio de features
   - Votación por mayoría (clasificación) o promedio (regresión)

2. **Reduce varianza sin aumentar bias**
   - Árboles individuales tienen alta varianza (inestables)
   - Promediar decorrelaciona errores
   - Resultado: modelo más robusto y generalizable

3. **OOB Error proporciona validación gratuita**
   - ~37% de ejemplos no vistos por cada árbol
   - Permite evaluar sin validation set separado
   - Muy útil con datasets pequeños

4. **Feature Importances para interpretabilidad**
   - Mide contribución de cada feature
   - Útil para feature selection
   - Promediado sobre todos los árboles = más estable

5. **Hiperparámetros clave:**
   - `n_estimators`: más árboles = mejor (pero rendimientos decrecientes)
   - `max_features`: controla decorrelación (típicamente √p)
   - `max_depth`, `min_samples_split`: controlan overfitting individual

6. **Ventajas:**
   - Excelente out-of-the-box performance
   - Poco tuning de hiperparámetros necesario
   - Maneja features de diferentes escalas
   - Robusto a outliers
   - Parallelizable

7. **Desventajas:**
   - Menos interpretable que un árbol individual
   - Más lento para predicción que modelos lineales
   - Puede consumir mucha memoria con muchos árboles profundos

---

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

- **"Random Forests"** - Leo Breiman (2001)
  - [Paper original](https://link.springer.com/article/10.1023/A:1010933404324)
  - El paper seminal que introdujo Random Forests
  - Explica teóricamente por qué funciona la decorrelación

- **"Bagging Predictors"** - Leo Breiman (1996)
  - Introduce el concepto de bootstrap aggregating
  - Base teórica para Random Forests

#### 📖 Libros Recomendados

- **"The Elements of Statistical Learning"** - Hastie, Tibshirani, Friedman
  - Capítulo 15: Random Forests
  - Análisis matemático riguroso
  - Disponible gratis: https://web.stanford.edu/~hastie/ElemStatLearn/

- **"Introduction to Statistical Learning"** - James, Witten, Hastie, Tibshirani
  - Capítulo 8.2: Bagging, Random Forests, Boosting
  - Más accesible, con ejemplos en R

#### 🎥 Videos Recomendados

- **StatQuest: Random Forests** - Josh Starmer
  - Explicación visual excelente
  - https://www.youtube.com/watch?v=J4Wdy0Wc_xQ

- **Andrew Ng - Ensemble Methods**
  - Machine Learning Course (Coursera)
  - Contexto de ensemble learning

#### 💻 Documentación y Recursos

- [Scikit-learn: Random Forest](https://scikit-learn.org/stable/modules/ensemble.html#forest)
- [Scikit-learn: User Guide - Ensemble Methods](https://scikit-learn.org/stable/modules/ensemble.html)
- [Case Study: Kinect Body Part Recognition](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/BodyPartRecognition.pdf)
  - Aplicación famosa de Random Forests en producción

#### 🔬 Investigación Avanzada

- **Extremely Randomized Trees (Extra Trees)**
  - Más aleatorización → más decorrelación
  - Geurts et al. (2006)

- **Isolation Forests**
  - Uso de Random Forests para detección de anomalías
  - Liu et al. (2008)

---

### 🤔 Preguntas para Reflexionar

1. **¿Por qué es importante la decorrelación entre árboles?**
   - Pista: Piensa en la fórmula de varianza con correlación

2. **¿Cuándo usarías Random Forest vs un árbol individual?**
   - Considera interpretabilidad vs accuracy

3. **¿Cómo afecta el tamaño del dataset al número óptimo de árboles?**
   - Piensa en overfitting y tiempo de cómputo

4. **¿Random Forests puede sufrir de underfitting?**
   - ¿Qué pasa si los árboles individuales son muy simples?

---

## ➡️ Próximo Paso

En el siguiente notebook, **07. Boosting**, aprenderemos sobre:

- **Boosting vs Bagging**: Diferencias fundamentales
- **AdaBoost**: El primer algoritmo de boosting exitoso
- **Gradient Boosting**: Marco general para boosting
- **XGBoost y LightGBM**: Implementaciones modernas ultra-optimizadas
- **Regularización en boosting**: Control de overfitting

**Diferencia clave:** Random Forests entrena árboles en **paralelo** (independientes), Boosting los entrena **secuencialmente** (cada árbol corrige errores del anterior).

---

<div align="center">

**🌳 ¡De árboles individuales a bosques! 🌳**

**Continúa con: [07. Boosting](07-boosting.ipynb)**

[← 05. Árboles de Decisión](05-arboles-decision.ipynb) | [07. Boosting →](07-boosting.ipynb)

</div>